In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import numpy as np
import os
import pandas as pd
from collections import defaultdict
from constants import WORD_TYPES, PROPER_NOUN_TYPES
pd.options.display.max_columns = 100
pd.options.display.max_rows = 130

In [3]:
from utils_shared_hanzi_shorts import (
    enhance_data_settings, get_filtered_words, load_video_configs,
    get_character_counts,
    stitch_audios, draw_vocab_list_whole_image, 
    create_video_with_highlights, create_video_without_highlights
)
from utils_shorts import (
    create_directories
)
from utils_data import (load_raw_data, check_dups)

# Settings

In [4]:
audio_settings = {
    'voice_name_zh': 'zh-CN-XiaoxiaoNeural',
    'audio_plan': 'ctitle_c2word',
    'pause_ms_beginning': 150,
    'pause_ms_within_word': 200,
    'pause_ms_between': 500,
}

data_settings = {
    'shared_char': '口',
    'char_pinyin': 'kǒu',
    'char_english': 'mouth; opening',
    'max_priority': 5,
    'min_adu': 3,
    'min_per': 3,
    'types_allowed': WORD_TYPES + PROPER_NOUN_TYPES,
    'sort_cols': ['priority', 'cat_v3', 'pinyin'],
    'sort_ascending': [True, True, True],
    'words_rmv': ['进口出口', '重口味', '壶口瀑布', '单口喜剧', '口腔科', '吐口水'],
    'n_words_per_video': 10,
    'current_part': 1,
    'text_replacements': {
        'exit;to export': 'exit; export',
        'gap;shortfall;insufficiency': 'gap;insufficiency',
        'stand up comedy;talk show': 'talk show;stand-up',
        'intersection;road crossing': 'roads intersection',
        'to speak;start talking;open mouth': 'to start talking'
        }
}

video_configs = load_video_configs()

In [5]:
data_settings = enhance_data_settings(data_settings)
create_directories(data_settings)
data_settings

{'shared_char': '口',
 'char_pinyin': 'kǒu',
 'char_english': 'mouth; opening',
 'max_priority': 5,
 'min_adu': 3,
 'min_per': 3,
 'types_allowed': ['word',
  'prefix',
  'suffix',
  'abbreviation',
  'multi_word',
  'verb_ending',
  'proper noun'],
 'sort_cols': ['priority', 'cat_v3', 'pinyin'],
 'sort_ascending': [True, True, True],
 'words_rmv': ['进口出口', '重口味', '壶口瀑布', '单口喜剧', '口腔科', '吐口水'],
 'n_words_per_video': 10,
 'current_part': 1,
 'text_replacements': {'exit;to export': 'exit; export',
  'gap;shortfall;insufficiency': 'gap;insufficiency',
  'stand up comedy;talk show': 'talk show;stand-up',
  'intersection;road crossing': 'roads intersection',
  'to speak;start talking;open mouth': 'to start talking'},
 'output_path_base': 'output/shared_char_shorts/',
 'output_path': 'output/shared_char_shorts/口',
 'output_path_audio': 'output/shared_char_shorts/口/audio_files',
 'output_path_images': 'output/shared_char_shorts/口/images'}

# Load data

In [6]:
truly_load_data = False
df_all_vocab = load_raw_data(truly_load_data=truly_load_data)
df_dups = check_dups(df_all_vocab)
print(df_all_vocab.shape)
print(f'# duplicate vocab: {len(df_dups)}')
df_all_vocab.head(3)

!!!!!!!! WARNING: not truly loading data !!!!!!!!
(7488, 38)
# duplicate vocab: 0


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,1,房贷,fáng dài,mortgage,word,1.0,life,NaN,Finance & Economy,NaN,NaN,NONE,1.0,1.0,2.0,1.0,房子,house,贷款,loan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他每个月都要还房贷,Ta měi gè yuè dōu yào huán fángdài,He has to pay his mortgage every month,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,2,白天,bái tiān,daytime,word,2.0,time,NaN,Time,NaN,NaN,1,2.0,1.0,1.0,1.0,白,white,天,day,NaN,NaN,NaN,NaN,NaN,NaN,NaN,白天很热晚上比较凉快,Báitiān hěn rè wǎnshàng bǐjiào liángkuai,It is hot in the daytime and cooler at night,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,3,组成,zǔ chéng,to form;make up,word,3.0,general,NaN,Language & Expression,NaN,NaN,2,5.0,5.0,5.0,3.0,组,set,成,become,NaN,NaN,NaN,NaN,NaN,NaN,NaN,水是由氢和氧组成的,Shuǐ shì yóu qīng hé yǎng zǔchéng de,Water is made up of hydrogen and oxygen,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


In [7]:
df_character_appearances = get_character_counts(df_all_vocab, data_settings)
print(len(df_all_vocab), len(df_character_appearances))
df_character_appearances['character'].value_counts().head(20)

7488 9410


character
子    70
大    62
人    61
机    44
电    37
心    36
不    35
口    35
地    34
手    32
车    32
水    32
时    31
自    31
花    31
生    31
动    31
中    30
小    29
外    29
Name: count, dtype: int64

In [8]:
df_filt = get_filtered_words(df_all_vocab, data_settings)
print([data_settings['shared_char']] + df_filt['chinese'].values.tolist())
print(len(df_filt))
df_filt.head(40)

['口', '出口', '口罩', '口音', '进口', '口红', '借口', '脱口秀', '口味', '口香糖', '胃口', '口水', '口语', '路口', '人口', '户口', '流口水', '口疮', '伤口', '口号', '口吻', '绕口令', '登机口', '口译', '口琴', '口吃', '口腔', '开口', '缺口']
28


/Users/scott/Development/my_mandarin_database/utils_shared_hanzi_shorts.py:136: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_filtered = df_filtered.replace(data_settings['text_replacements'])


,id,chinese,pinyin,english,type,priority,category1,category2,cat_v3,cat2_v3,cat3_v3,hsk_level,known,known_pinyin_prompt,known_english_prompt,quality,word1,word1_english,word2,word2_english,word3,word3_english,word4,word4_english,voice_zh,voice_en,video_notes,sentence,sentence_pinyin,sentence_english,date,source1,source2,funny,per,adu,slang,phonetic
0,351,出口,chū kǒu,exit;export,word,1.0,society,NaN,Business & Shopping,NaN,NaN,2,1.0,1.0,1.0,3.0,出,to go out,口,opening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,请问出口在哪里,qǐng wèn chū kǒu zài nǎ lǐ,excuse me where is the exit,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
1,342,口罩,kǒu zhào,mask,word,1.0,health,supplies,Health & Body,NaN,NaN,7-9,5.0,2.0,2.0,2.0,口,mouth,胸罩,bra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,出门记得戴口罩,Chūmén jìdé dài kǒuzhào,Remember to wear a mask when going out,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
2,6351,口音,kǒu yīn,accent,word,1.0,NaN,NaN,Linguistics,NaN,NaN,MISSING,5.0,5.0,5.0,1.0,口,mouth,音调,pitch;tone,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他的四川口音特别明显。,tā de sìchuān kǒuyīn tèbié míngxiǎn.,His Sichuan accent is very noticeable.,2025-12-12,daily add,NaN,NaN,5.0,5.0,5.0,NaN
3,354,进口,jìn kǒu,import,word,2.0,career,Business & Finance,Business & Shopping,NaN,NaN,4,3.0,2.0,5.0,2.0,进,enter,口,opening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这酒是进口的,Zhè jiǔ shì jìnkǒu de,This wine is imported,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
4,1232,口红,kǒu hóng,lipstick,word,2.0,clothes,Cosmetics & Makeup,Clothes & Accessories,NaN,配饰与穿戴用品,NONE,2.0,1.0,2.0,1.0,口,mouth,红,red,NaN,NaN,NaN,NaN,NaN,NaN,NaN,她买了一支口红,Tā mǎi le yī zhī kǒuhóng,She bought a lipstick,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
5,3889,借口,jiè kǒu,excuse,word,2.0,people,NaN,Emotions & Relationships,NaN,NaN,7-9,5.0,2.0,5.0,2.0,借,to borrow,口,mouth,NaN,NaN,NaN,NaN,NaN,NaN,NaN,他总找借口不做作业,Tā zǒng zhǎo jièkǒu bù zuò zuòyè,He always finds excuses not to do homework,2025-07-08,daily add,NaN,NaN,5.0,5.0,5.0,NaN
6,6975,脱口秀,tuō kǒu xiù,talk show;stand-up,word,2.0,NaN,NaN,Entertainment,NaN,NaN,MISSING,5.0,5.0,5.0,3.0,脱掉,take off,口,mouth,秀,show,NaN,NaN,NaN,NaN,NaN,他最近开始尝试脱口秀表演。,tā zuì jìn kāi shǐ cháng shì tuō kǒu xiù biǎo ...,he recently started trying stand up comedy.,2026-01-17,daily add,NaN,NaN,5.0,5.0,5.0,NaN
7,2026,口味,kǒu wèi,taste,word,2.0,food,NaN,Food & Drink,NaN,面食与米制品,7-9,2.0,1.0,2.0,2.0,口,mouth,味,flavor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,这家冰淇淋有很多口味,Zhè jiā bīngqílín yǒu hěn duō kǒuwèi,This ice cream shop has many flavors,2025-01-02,menus,NaN,NaN,5.0,5.0,5.0,NaN
8,399,口香糖,kǒu xiāng táng,gum,word,2.0,food,NaN,Food & Drink,NaN,食品加工与安全,7-9,2.0,1.0,2.0,1.0,口,mouth,香,fragrant,糖,sugar,NaN,NaN,NaN,NaN,NaN,他正在嚼口香糖,Tā zhèngzài jiáo kǒuxiāngtáng,He is chewing gum,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN
9,783,胃口,wèi kǒu,appetite,word,2.0,food,NaN,Food & Drink,NaN,咖啡与咖啡文化,7-9,2.0,1.0,4.0,1.0,胃,stomach,口,mouth,NaN,NaN,NaN,NaN,NaN,NaN,NaN,我今天没胃口,Wǒ jīntiān méi wèikǒu,I don’t have an appetite today,2025-01-02,NaN,NaN,NaN,5.0,5.0,5.0,NaN


# Create video for each part

In [9]:
print('Cut vocab into parts')
data_settings['n_words_total'] = len(df_filt)
data_settings['n_parts'] = int(np.ceil(len(df_filt) / data_settings['n_words_per_video']))

for current_part in range(1, data_settings['n_parts'] + 1):
    # Determine vocabulary in current part
    data_settings['current_part'] = current_part
    start_index = (current_part - 1) * data_settings['n_words_per_video']
    end_index = start_index + data_settings['n_words_per_video'] - 1
    data_settings['current_part_index_range'] = (start_index, end_index)
    print(f"Processing part {current_part}/{data_settings['n_parts']} with index range {data_settings['current_part_index_range']}")
    df_filt_currentpart = df_filt[
        (df_filt.index >= data_settings['current_part_index_range'][0]) &
        (df_filt.index <= data_settings['current_part_index_range'][1])
    ].reset_index(drop=True)

    print('Making audio')
    df_durations = stitch_audios(audio_settings, data_settings, df_filt_currentpart['chinese'].values.tolist())
    print('Making image')
    no_hl_img_file_path = draw_vocab_list_whole_image(video_configs, data_settings, df_filt_currentpart)
    print('Making video without highlights')
    create_video_without_highlights(data_settings, video_configs, no_hl_img_file_path)
    print('Making video with highlights')
    create_video_with_highlights(df_durations, audio_settings, data_settings, video_configs)

Cut vocab into parts
Processing part 1/3 with index range (0, 9)
Making audio
Audio duration: 34.2s
Making image
Making video without highlights
MoviePy - Building video output/shared_char_shorts/口/口_no_highlights_part1.mp4.
MoviePy - Writing audio in 口_no_highlights_part1TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/口/口_no_highlights_part1.mp4



MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/口/口_no_highlights_part1.mp4
Making video with highlights
MoviePy - Building video output/shared_char_shorts/口/口_part1.mp4.
MoviePy - Writing audio in 口_part1TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/口/口_part1.mp4



frame_index: 100%|██████████| 822/822 [00:17<00:00, 48.49it/s, now=None]/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file output/shared_char_shorts/口/口_no_highlights_part1.mp4, 2764800 bytes wanted but 0 bytes read at frame index 821 (out of a total 821 frames), at time 34.21/34.22 sec. Using the last valid frame instead.
  warnings.warn(


...

MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/口/口_no_highlights_part3.mp4



MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/口/口_no_highlights_part3.mp4
Making video with highlights
MoviePy - Building video output/shared_char_shorts/口/口_part3.mp4.
MoviePy - Writing audio in 口_part3TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video output/shared_char_shorts/口/口_part3.mp4



frame_index: 100%|█████████▉| 674/677 [00:14<00:00, 46.16it/s, now=None]/Users/scott/Library/Caches/pypoetry/virtualenvs/my-mandarin-database-XyoBA6Fv-py3.13/lib/python3.13/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file output/shared_char_shorts/口/口_no_highlights_part3.mp4, 2764800 bytes wanted but 0 bytes read at frame index 676 (out of a total 676 frames), at time 28.17/28.19 sec. Using the last valid frame instead.
  warnings.warn(


MoviePy - Done !
MoviePy - video ready output/shared_char_shorts/口/口_part3.mp4
